## Librairies

In [1]:
import numpy as np
import cvxpy as cp
import pandas as pd
from tqdm import tqdm
import plotly.express as px
from scipy.optimize import minimize
from python_module.pricing_model import BSMModel

pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
pd.options.display.float_format = '{:,.4f}'.format

## Functions

In [ ]:
def compute_vt_price(price_ts, exp_window, target_vol, init_vt_price = 100):
    price_df = price_ts.to_frame(name='risky_asset')
    log_returns_df = np.log(price_df).diff()
    rolling_std_df = log_returns_df.ewm(span=exp_window, min_periods=exp_window, adjust=False).std()
    rolling_std_df *= (252 ** 0.5)
    leverage_df = target_vol / rolling_std_df
    vt_price_df = (price_df.loc[leverage_df.dropna().index].pct_change() * leverage_df.shift(1)).fillna(0).add(1).cumprod() * init_vt_price
    vt_price_df.columns = ['F']
    agg_df = pd.concat([price_df, leverage_df['risky_asset'].to_frame('beta'), vt_price_df], axis=1).dropna()
    agg_df['vt_delta'] = (agg_df['F'] * agg_df['beta'])/agg_df['risky_asset']
    return agg_df

def apply_compute_option(row):
    inputs = row[['F', 'K', 'T', 'r', 'sigma', 'option_type', 'compute_greeks', 'slide_scenario', 'slide_compute']].to_dict()
    return BSMModel.compute_option(**inputs)

def compute_vt_option_bt(agg_df, day_to_maturity, strike_delta, target_vol, slide_scenario):
    
    results = list()
    for date in tqdm(agg_df.index):
        agg_df_temp = agg_df.loc[date:].iloc[:day_to_maturity+1]
        if agg_df_temp.shape[0] != day_to_maturity+1: break
        agg_df_temp['strike_date'] = agg_df_temp.index[0]
        agg_df_temp['maturity_date'] = agg_df_temp.index[-1]
        agg_df_temp['days_to_maturity'] = list(range(agg_df_temp.shape[0]))[::-1]
        agg_df_temp['T'] = agg_df_temp['days_to_maturity'] / 252

        option_type = 'call' if strike_delta > 0 else 'put'
        agg_df_temp['option_type'] = option_type

        strike_k = BSMModel.solve_delta_strike(F=100, T=day_to_maturity/252, sigma=target_vol, r=0, option_type=option_type, target_delta=strike_delta)
        strike_pct = strike_k / 100
        agg_df_temp['K'] = agg_df_temp['F'].iloc[0] * strike_pct
        agg_df_temp['r'] = 0
        agg_df_temp['sigma'] = target_vol
        agg_df_temp['compute_greeks'] = True
        agg_df_temp['slide_scenario'] = slide_scenario
        agg_df_temp['slide_compute'] = 'option_pnl'
        
        pricing_df = agg_df_temp.apply(lambda x: apply_compute_option(x), axis=1, result_type='expand')
        agg_df_temp = pd.concat([agg_df_temp, pricing_df], axis=1)
        agg_df_temp['asset_delta'] = agg_df_temp['vt_delta'] * agg_df_temp['delta']
        agg_df_temp['asset_delta_cash'] = agg_df_temp['risky_asset'] * agg_df_temp['asset_delta']
        agg_df_temp['dP'] = agg_df_temp['price'].diff()
        agg_df_temp['dH'] = agg_df_temp['risky_asset'].diff() * agg_df_temp['asset_delta'].shift(1)
        results.append(agg_df_temp)
    results_df = pd.concat(results)
    return results_df

def compute_scaling(trading_units, trading_scale, results_df, slide_scenario, day_to_maturity, cols=['asset_delta_cash','dP', 'dH']):
    scaling_factor = trading_units / results_df.groupby('strike_date')[trading_scale].first()
    results_df['scaling_factor'] = results_df['strike_date'].map(scaling_factor)
    for col in cols:
        results_df[f'scaled_{col}'] = results_df['scaling_factor'] * results_df[col]
    results_df[f'scaled_{trading_scale}'] = results_df['scaling_factor'] * results_df[trading_scale]
    results_df[f'scaled_{slide_scenario}'] = results_df['scaling_factor'] * results_df[slide_scenario]
    nb_strike = results_df.groupby(results_df.index)['strike_date'].count()
    mask = (nb_strike == (day_to_maturity+1))
    results_df = results_df.loc[mask]
    return results_df


## Inputs

In [ ]:
pairs = [['SPY', 'QQQ'], ['EWQ', 'EWG']]

exp_window = 20
target_vol = 0.10
strike_delta = -0.1
day_to_maturity = 250
slide_scenario = -0.3

# scaling
trading_units = 20 * (10 / day_to_maturity)
trading_scale = 'theta' # 'F', 'price', 'delta', 'gamma', 'vega', 'theta', 'vanna', 'volga', slide_scenario

## Script

In [ ]:
# load daily returns
gross_daily_pnl = dict()
for pair in pairs:
    for symbol in pair:
        print(f'loading symbol {symbol}')
        price_ts = pd.read_csv(f'data/{symbol}.csv', index_col=0, parse_dates=True)['price']
        agg_df = compute_vt_price(price_ts, exp_window, target_vol)
        results_df = compute_vt_option_bt(agg_df, day_to_maturity, strike_delta, target_vol, slide_scenario)
        scaled_res_df = compute_scaling(trading_units, trading_scale, results_df, slide_scenario, day_to_maturity)
        gross_daily_pnl[symbol] = scaled_res_df.groupby(scaled_res_df.index)['scaled_dH'].sum()
gross_daily_pnl_df = pd.DataFrame(gross_daily_pnl)

In [ ]:
gross_daily_pnl_df.describe()

In [ ]:
px.line(gross_daily_pnl_df.cumsum())

In [ ]:
for pair in pairs:

    print(f"pair: {pair}")
    returns = gross_daily_pnl_df.loc[:, pair]

    mu = returns.mean() * 0
    mu = mu+1
    
    sigma = returns.cov() * 252
    n_assets = len(mu)

    inv_sigma = np.linalg.inv(sigma)

    # Calculate unscaled weights (z)
    z = np.dot(inv_sigma, mu)

    # Normalize so sum(weights) = 1
    w_opt = z / np.sum(z)
    print(f"weights: {w_opt}")
    display(px.line(returns.dot(w_opt).dropna().cumsum()))
    strategy_returns = returns.dot(w_opt).dropna()
    print('display returns')
    display(strategy_returns.groupby(strategy_returns.index.year).sum())